In [1]:
import os
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import pandas as pd

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
model_id = "Qwen/Qwen3-VL-8B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto",
)

model.eval()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [3]:
base_dir = "./Version0_dataset"

folders = [
    "Qwen_Image_Edit_2509_v0",
    "Qwen_Image_Edit_2509_v0_armchair",
    "Qwen_Image_Edit_2509_v0_table"
]

image_paths = []

for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    for file in os.listdir(folder_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(folder_path, file))

print("Total images:", len(image_paths))


Total images: 630


In [4]:
def ask_yes_no(img_path, question):
    prompt = f"{question} Answer with exactly one word: yes or no."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": "file://" + os.path.abspath(img_path)},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    images, videos = process_vision_info(messages, image_patch_size=16)

    inputs = processor(text=[text], images=images, videos=videos, do_resize=False, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    # print("pixel_values:", inputs["pixel_values"].shape, inputs["pixel_values"].dtype) # pixel_values: torch.Size([4144, 1536]) torch.float32
    # if "image_grid_thw" in inputs:
    #     print("image_grid_thw:", inputs["image_grid_thw"].shape, inputs["image_grid_thw"][:2]) # image_grid_thw: torch.Size([1, 3]) tensor([[ 1, 56, 74]], device='cuda:0')


    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
        )
        # do_sample=False, ##. ## try first generate sth correct, then try generate logis\ts / probs
        # use_cache=True,
    

    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], output)]
    response = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return response.strip().lower()


In [ ]:
import torch
from PIL import Image

def ask_yes_no_PIL_input(img_path, question):
    # 1. 每次出错后必须重启 kernel，因为 CUDA assert 是致命的
    # 2. 读取图片并确保是 RGB
    image = Image.open(img_path).convert("RGB")
    
    # 3. 构造标准消息（注意这里直接传 PIL 对象）
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image}, 
                {"type": "text", "text": f"{question} Answer with exactly one word: yes or no."},
            ],
        }
    ]

    # 4. 让 processor 一站式处理模板、缩放和对齐
    # 这样它会自动帮你处理掉那个恼人的 shape 校验问题
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # 注意：这里不传 images=images，而是传最原始的 messages
    # Processor 会自动调用内部逻辑去 resize
    inputs = processor(
        text=[text], 
        images=[image], 
        return_tensors="pt"
    ).to(model.device)

    # 5. 生成
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
        )

    # 6. 解码
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, output)
    ]
    response = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]

    return response.strip().lower()

In [ ]:
# test one image
questions_template = [
    "Is there only 1 human in this image?",
    "Is the {second_object} next to the human?",
    "Is the {second_object} clearly recognisable?",
    "Is there any artefact?"
]

for img_path in image_paths:
    print("="*80)
    print("Image:", img_path)

    # image = Image.open(img_path).convert("RGB")

    filename = os.path.basename(img_path)
    second_object = filename.split("_", 1)[0]

    for q_template in questions_template:
        question = q_template.format(second_object=second_object)

        answer = ask_yes_no(img_path, question)

        print(f"Q: {question}")
        print(f"A: {answer}")
        print()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Image: ./Version0_dataset/Qwen_Image_Edit_2509_v0/bottle-of-orange-juice_left_of_chair_FACE-CAMERA.png
pixel_values: torch.Size([4144, 1536]) torch.float32
image_grid_thw: torch.Size([1, 3]) tensor([[ 1, 56, 74]], device='cuda:0')
Q: Is there only 1 human in this image?
A: yes

pixel_values: torch.Size([4144, 1536]) torch.float32
image_grid_thw: torch.Size([1, 3]) tensor([[ 1, 56, 74]], device='cuda:0')
Q: Is the bottle-of-orange-juice next to the human?
A: yes

pixel_values: torch.Size([4144, 1536]) torch.float32
image_grid_thw: torch.Size([1, 3]) tensor([[ 1, 56, 74]], device='cuda:0')
Q: Is the bottle-of-orange-juice clearly recognisable?
A: yes

pixel_values: torch.Size([4144, 1536]) torch.float32
image_grid_thw: torch.Size([1, 3]) tensor([[ 1, 56, 74]], device='cuda:0')
Q: Is there any artefact?
A: yes

Image: ./Version0_dataset/Qwen_Image_Edit_2509_v0/blanket_right_of_chair_FACE-LEFT.png
pixel_values: torch.Size([4144, 1536]) torch.float32
image_grid_thw: torch.Size([1, 3]) tenso

KeyboardInterrupt: 

In [12]:
questions_template = [
    "Is there only 1 human in this image?",
    "Is the {second_object} next to the human?",
    "Is the {second_object} in the image?",
    "Is the {second_object} clearly recognisable?",
]
output_csv = "validation_qwen3vl.csv"

import csv
with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    header = [
        "image_path",
        "only_one_human",
        "object_next_to_human",
        "object_in_image",
        "object_recognisable",
    ]
    writer.writerow(header)

    for img_path in image_paths:
        # print("=" * 80)
        # print("Image:", img_path)

        filename = os.path.basename(img_path)
        second_object = filename.split("_", 1)[0]

        answers = []

        for q_template in questions_template:
            question = q_template.format(second_object=second_object)
            answer = ask_yes_no(img_path, question)

            # print(f"Q: {question}")
            # print(f"A: {answer}")
            # print()
            if answer != "yes":
                print(f"{question} not yes: {img_path}")

            answers.append(answer)

        row = [img_path] + answers
        writer.writerow(row)

        # for idx, ans in enumerate(answers):
        #     if ans != "yes":
        #         print(f"{idx}: {question} not yes: {img_path}")

print("Results saved to", output_csv)


Is the mug next to the human? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/mug_left_of_chair_FACE-LEFT.png
Is the mug in the image? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/mug_left_of_chair_FACE-LEFT.png
Is the mug clearly recognisable? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/mug_left_of_chair_FACE-LEFT.png
Is the painting clearly recognisable? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/painting_left_of_chair_FACE-LEFT.png
Is the ball-of-yarn clearly recognisable? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/ball-of-yarn_right_of_chair_FACE-CAMERA.png
Is the shoes next to the human? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/shoes_left_of_chair_FACE-LEFT.png
Is the remote next to the human? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/remote_left_of_chair_FACE-RIGHT.png
Is the book clearly recognisable? not yes: ./Version0_dataset/Qwen_Image_Edit_2509_v0/book_left_of_chair_FACE-RIGHT.png
Is the beer-bottle clearly recognisa

In [13]:
# second validation round: complete human figure
questions_template = [
    "Is the human figure complete in the image?",
    # "Is the entire human figure fully inside the image frame (no body parts cut off by the border)?", # not enough
    # "Is the very top of the head fully visible (no missing hair/forehead due to cropping)?" # too much
    # "Is any part of the head actually cut off by the image border?"
    "Is there any margin between the top of the head and the upper border of the image?"
]
output_csv_2 = "validation_qwen3vl_2nd.csv"

validation_1st = pd.read_csv("validation_qwen3vl.csv").set_index("image_path")
print("First round validation results loaded: len=", len(validation_1st))
valid_1st_count =0
for img_path in validation_1st.index:
    row = validation_1st.loc[img_path]
    if row["object_next_to_human"] != "yes" and row["object_recognisable"] != "yes" and row["object_in_image"] != "yes":
        continue
    else:
        valid_1st_count += 1
print("Valid images from first round: len=", valid_1st_count)


valide_image_paths = []

import csv
with open(output_csv_2, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    header = [
        "image_path",
        "human_figure_complete",
        "human_body_cut_off",
    ]
    writer.writerow(header)

    for i, img_path in enumerate(image_paths):

        if img_path in validation_1st.index:
            row = validation_1st.loc[img_path]
            if row["object_next_to_human"] != "yes" and row["object_recognisable"] != "yes" and row["object_in_image"] != "yes":
                continue
            else:
                # filename = os.path.basename(img_path)
                # second_object = filename.split("_", 1)[0]

                answers = []

                for q_template in questions_template:
                    question = q_template.format(second_object=second_object)
                    answer = ask_yes_no(img_path, question)
                    answers.append(answer)
                    # print(f"Q: {question}")
                    # print(f"A: {answer}")
                    # print()
                    # if answer != "yes":
                    #     print(f"{question} not yes: {img_path}")
                    # else:
                    #     valide_image_paths.append(img_path)
                if answers[0] == "yes" and answers[1] == "yes":
                    valide_image_paths.append(img_path)
                print(f"{questions_template[0]} not yes: {img_path}" if answers[0] != "yes" else "")
                print(f"{questions_template[1]} not yes: {img_path}" if answers[1] != "yes" else "")

                row = [img_path] + answers
                writer.writerow(row)

                # for idx, ans in enumerate(answers):
                #     if ans != "yes":
                #         print(f"{idx}: {question} not yes: {img_path}")

print("Results saved to", output_csv_2)
print("Valid image paths count:", len(valide_image_paths))

import json
with open("valide_image_paths.json", "w", encoding="utf-8") as f:
    json.dump(valide_image_paths, f, ensure_ascii=False, indent=4)


First round validation results loaded: len= 630
Valid images from first round: len= 567










































































































































































































































































































































































































































































































































































































































































































































































































































































































Is the human figure complete in the im